<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase27_FIXED_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICHI 2027 — Phase 27
## Pinned OMOP v5.5 + MI-CDM Revision Audit and Faithful Synthetic Arm A Gate
**Google Drive → Google Drive**

### Why this phase exists
Phase 26 established a clean synthetic FHIR R4 evidence layer and showed that the required neuroimaging evidence dimensions are reconstructable. Phase 27 now freezes the target model before any OMOP/MI-CDM claim is made.

### Phase 27 rules
- Pin **OMOP CDM v5.5** as the OMOP target family.
- Clone and pin the exact current **OHDSI/ImageWG** repository commit used by this run.
- Discover `IMAGE_OCCURRENCE` and `IMAGE_FEATURE` definitions from that pinned repository.
- Do **not** invent MI-CDM columns.
- Build a synthetic Arm A mapping only when the required target columns can be verified from the pinned source.
- Keep Arm C recorded as nonportable and unexecuted.
- No real ADNI rows.
- No Athena concept approval claim.
- No official MI-CDM production-load claim.


### Input-read fix (revised)

This revision avoids the `p.stat()` crash when Google Drive exposes a stale/moved
entry. It reconnects Drive, reads actual candidate files safely, and pairs the
Phase 26 shareable JSON with the **exact synthetic FHIR R4 bundle whose SHA-256
matches its recorded bundle hash**. Phase 24 is a prerequisite of earlier
phases; it is not the required Phase 27 input.

If Drive only contains Phase 24, run Phase 25 and Phase 26 or copy both existing
Phase 26 outputs into My Drive. If you know the location, set
`PHASE26_FOLDER_HINT` in the input cell below.


In [1]:

# Reconnect Google Drive to clear stale mount information.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:

# Phase 26 input discovery: safe against stale/moved Google Drive entries.
# You can override either individual file or the folder if automatic search misses them.
from pathlib import Path
import os, json, hashlib, subprocess, re, shutil, csv
from datetime import datetime, timezone
import pandas as pd

MYDRIVE = Path('/content/drive/MyDrive')

# OPTIONAL: set these to the CURRENT paths in My Drive (otherwise leave blank).
PHASE26_FOLDER_HINT = '/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs'
PHASE26_JSON_OVERRIDE = ''
PHASE26_BUNDLE_OVERRIDE = ''

def _digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def _scan(base):
    '''Scan filenames, then *open* each file rather than stat() stale entries.'''
    json_paths, bundle_paths = [], []
    if not base.is_dir():
        return json_paths, bundle_paths
    def onerror(err):
        print('Skipping inaccessible Drive directory:', err)
    for folder, _, names in os.walk(str(base), onerror=onerror):
        for name in names:
            path = Path(folder) / name
            if name.startswith('ICHI2027_Phase26_SHAREABLE') and name.endswith('.json'):
                json_paths.append(path)
            elif name.startswith('ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE') and name.endswith('.json'):
                bundle_paths.append(path)
    return json_paths, bundle_paths

def _read_valid_phase26(path):
    try:
        with open(path, encoding='utf-8') as f:
            obj = json.load(f)
        if obj.get('phase') != '26_synthetic_fhir_r4_evidence_contract_gate':
            return None
        expected = obj.get('fhir_representation', {}).get('synthetic_bundle_sha256')
        if not expected:
            print('Skipping Phase 26 JSON without bundle SHA:', path)
            return None
        return obj
    except (OSError, ValueError, TypeError) as exc:
        print('Skipping missing, stale, or unreadable Drive entry:', path, exc)
        return None

def _choose_pair(json_paths, bundle_paths):
    verified_jsons = [(p, _read_valid_phase26(p)) for p in json_paths]
    verified_jsons = [(p, d) for p, d in verified_jsons if d is not None]
    valid_bundles = []
    for p in bundle_paths:
        try:
            digest = _digest(p)
            with open(p, encoding='utf-8') as f:
                bundle = json.load(f)
            if bundle.get('resourceType') == 'Bundle' and bundle.get('type') == 'collection':
                valid_bundles.append((p, digest))
        except (OSError, ValueError, TypeError) as exc:
            print('Skipping missing or unreadable FHIR bundle:', p, exc)

    # Prefer a matching file in the same folder; never pair unrelated runs.
    pairs = [
        (jp, bp)
        for jp, obj in verified_jsons
        for bp, digest in valid_bundles
        if digest == obj['fhir_representation']['synthetic_bundle_sha256']
    ]
    pairs.sort(key=lambda pair: (pair[0].parent == pair[1].parent, str(pair[0])), reverse=True)
    return pairs[0] if pairs else None, len(verified_jsons), len(valid_bundles)

# An explicit file path is respected. Otherwise, look in the folder first,
# then retry across My Drive if nothing usable is found.
jsons, bundles = [], []
if PHASE26_JSON_OVERRIDE.strip():
    jsons = [Path(PHASE26_JSON_OVERRIDE.strip())]
if PHASE26_BUNDLE_OVERRIDE.strip():
    bundles = [Path(PHASE26_BUNDLE_OVERRIDE.strip())]

if not (jsons and bundles):
    hinted_jsons, hinted_bundles = _scan(Path(PHASE26_FOLDER_HINT)) if PHASE26_FOLDER_HINT.strip() else ([], [])
    if not jsons: jsons = hinted_jsons
    if not bundles: bundles = hinted_bundles

selection, jcount, bcount = _choose_pair(jsons, bundles)
if selection is None:
    # Second pass: search the whole My Drive; do not call stat() on search hits.
    all_jsons, all_bundles = _scan(MYDRIVE)
    selection, jcount, bcount = _choose_pair(
        jsons if PHASE26_JSON_OVERRIDE.strip() else all_jsons,
        bundles if PHASE26_BUNDLE_OVERRIDE.strip() else all_bundles
    )

if selection is None:
    print(f'Valid Phase 26 shareable JSON files found: {jcount}')
    print(f'Readable Phase 26 synthetic FHIR bundles found: {bcount}')
    print('Drive root:', MYDRIVE)
    print('Folder hint:', PHASE26_FOLDER_HINT)
    raise FileNotFoundError(
        'Phase 27 needs BOTH ICHI2027_Phase26_SHAREABLE.json AND '
        'ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json from the SAME Phase 26 run. '
        'They must have a matching bundle SHA-256. Phase 24 JSON is not sufficient. '
        'Check the folder in Google Drive; if the bundle is missing, rerun Phase 26 '
        'with Runtime > Run all to save all three Phase 26 outputs, then rerun Phase 27.'
    )

PHASE26_JSON, PHASE26_BUNDLE = selection
WORKDIR = PHASE26_JSON.parent
print('Verified Phase 26 shareable:', PHASE26_JSON)
print('Verified matching FHIR bundle:', PHASE26_BUNDLE)
print('Phase 27 Drive output folder:', WORKDIR)


Verified Phase 26 shareable: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SHAREABLE.json
Verified matching FHIR bundle: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json
Phase 27 Drive output folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


In [3]:

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with open(PHASE26_JSON, "r", encoding="utf-8") as f:
    p26 = json.load(f)

assert p26["phase"] == "26_synthetic_fhir_r4_evidence_contract_gate"
assert p26["gates"]["phase25_verified"] is True
assert p26["gates"]["synthetic_FHIR_R4_representation_created"] is True
assert p26["gates"]["synthetic_required_evidence_reconstructable"] is True
assert p26["gates"]["synthetic_FHIR_contract_cases_passed"] is True
assert p26["gates"]["pyOMOP_ETL_executed"] is False
assert p26["gates"]["official_MICDM_load_completed"] is False
assert p26["gates"]["real_MRI_rows_processed"] == 0

assert sha256_file(PHASE26_BUNDLE) == p26['fhir_representation']['synthetic_bundle_sha256'], (
    'Phase 26 synthetic bundle SHA-256 mismatch: wrong or modified bundle.'
)
print("Phase 26 gate AND bundle hash verified.")
print("Phase 26 SHA256:", sha256_file(PHASE26_JSON))


Phase 26 gate AND bundle hash verified.
Phase 26 SHA256: 6cb3ef72e4d622d2a7330c7983c228f40a8e7aafecb5448395aace630f82bca4



## Freeze external targets

The current OHDSI CommonDataModel repository recommends v5.5 rather than v6.0 for transformation work. This notebook therefore records `OMOP CDM v5.5` as the target model family.

MI-CDM is treated differently: the OHDSI ImageWG repository is evolving. We therefore pin the **exact git commit used in this run** and inspect that revision for machine-readable or textual definitions of `IMAGE_OCCURRENCE` and `IMAGE_FEATURE`.

If the repository cannot be cloned or the schema cannot be verified, Phase 27 exits conservatively rather than fabricating a mapping.


In [4]:

# Clone/pin OHDSI ImageWG
IMAGEWG_DIR = Path("/content/OHDSI_ImageWG_phase27")
if IMAGEWG_DIR.exists():
    shutil.rmtree(IMAGEWG_DIR)

clone_cmd = [
    "git", "clone", "--depth", "1",
    "https://github.com/OHDSI/ImageWG.git",
    str(IMAGEWG_DIR)
]

clone = subprocess.run(clone_cmd, capture_output=True, text=True)
if clone.returncode != 0:
    raise RuntimeError(
        "Could not clone OHDSI/ImageWG. Phase 27 will not invent a target schema.\n"
        + clone.stderr[-2000:]
    )

commit = subprocess.check_output(
    ["git", "-C", str(IMAGEWG_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

print("Pinned ImageWG commit:", commit)


Pinned ImageWG commit: 8f0a19bb94e941bb335a097bc9851fab672621c7


In [5]:

# Inventory candidate files that mention the two MI-CDM imaging tables.
text_exts = {
    ".sql", ".csv", ".tsv", ".md", ".rmd", ".txt",
    ".json", ".yaml", ".yml", ".R", ".r", ".html"
}

hits = []
for path in IMAGEWG_DIR.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in text_exts:
        continue
    try:
        txt = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    low = txt.lower()
    if "image_occurrence" in low or "image_feature" in low:
        hits.append({
            "file": str(path.relative_to(IMAGEWG_DIR)),
            "mentions_image_occurrence": "image_occurrence" in low,
            "mentions_image_feature": "image_feature" in low,
            "sha256": hashlib.sha256(txt.encode("utf-8")).hexdigest(),
        })

schema_inventory = pd.DataFrame(hits).sort_values("file") if hits else pd.DataFrame(
    columns=["file","mentions_image_occurrence","mentions_image_feature","sha256"]
)
display(schema_inventory)

if schema_inventory.empty:
    raise RuntimeError(
        "Pinned ImageWG revision contains no inspectable IMAGE_OCCURRENCE/IMAGE_FEATURE references. "
        "Stopping rather than inventing schema fields."
    )


,file,mentions_image_occurrence,mentions_image_feature,sha256
0,README.md,True,True,cf308882e5616d408d76da1dcf535699df3463414bdaca...
5,docs/Approach.html,True,True,2076980749956d0c0ab4e1fc590f120e73dc18da047c70...
4,docs/background.html,True,True,20ae23e2edd8a487e9d66294e939aa68c0de57a5f18103...
6,inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_F...,True,True,75de1b0ccab328742110157d464070a3a2c8cb37342c25...
1,rmd/Approach.Rmd,True,True,f85d5711c5b41341676852f5df7b9f481b6b4e71956e37...
3,rmd/background.Rmd,True,True,068b7d5e9e296c391296f65d2ebbd3f78f81b10018afa0...
2,rmd/index.Rmd,True,True,db263b3925fa706bdfc4ead7855ca146b1c08b0d538e8f...


In [6]:

# Extract CREATE TABLE column definitions if present.
# Conservative parser: only accepts explicit CREATE TABLE blocks.

def extract_create_table_columns(text, table_name):
    pat = re.compile(
        rf"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?(?:[\w\.]+\.)?{table_name}\s*\((.*?)\)\s*;",
        flags=re.I | re.S
    )
    m = pat.search(text)
    if not m:
        return None
    body = m.group(1)

    cols = []
    depth = 0
    current = []
    parts = []
    for ch in body:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append("".join(current))
            current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))

    for p in parts:
        line = re.sub(r"--.*$", "", p.strip(), flags=re.M).strip()
        if not line:
            continue
        first = re.split(r"\s+", line)[0].strip('`"[]')
        if first.upper() in {
            "PRIMARY","FOREIGN","CONSTRAINT","UNIQUE","KEY","CHECK"
        }:
            continue
        cols.append(first)
    return cols

table_defs = {}
definition_sources = {}

for rec in hits:
    path = IMAGEWG_DIR / rec["file"]
    txt = path.read_text(encoding="utf-8", errors="ignore")
    for table in ["image_occurrence", "image_feature"]:
        cols = extract_create_table_columns(txt, table)
        if cols:
            table_defs[table] = cols
            definition_sources[table] = rec["file"]

print("Parsed explicit table definitions:")
for k,v in table_defs.items():
    print(k, "from", definition_sources[k], "->", v)


Parsed explicit table definitions:


In [7]:

# If explicit SQL DDL was not found, search CSV/schema-like files for table/field rows.
# We only accept fields actually present in the pinned repository.

def normalize(s):
    return re.sub(r"[^a-z0-9_]+", "", str(s).lower())

def csv_schema_candidates(repo_dir):
    candidates = []
    for path in repo_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".tsv"}:
            continue
        try:
            df = pd.read_csv(path, sep=None, engine="python")
        except Exception:
            continue
        cols = [normalize(c) for c in df.columns]
        whole = " ".join(cols + [normalize(x) for x in df.astype(str).head(200).values.ravel()])
        if "image_occurrence" in whole or "image_feature" in whole:
            candidates.append((path, df))
    return candidates

csv_candidates = csv_schema_candidates(IMAGEWG_DIR)

# Try common schema layouts only when explicit DDL was absent.
for table in ["image_occurrence", "image_feature"]:
    if table in table_defs:
        continue
    for path, df in csv_candidates:
        norm_cols = {normalize(c): c for c in df.columns}
        table_col = next((orig for n,orig in norm_cols.items() if n in {"table","tablename","cdmtable"}), None)
        field_col = next((orig for n,orig in norm_cols.items() if n in {"field","fieldname","column","columnname"}), None)
        if table_col and field_col:
            mask = df[table_col].astype(str).str.lower().str.replace(" ", "_", regex=False).eq(table)
            fields = [str(x) for x in df.loc[mask, field_col].dropna().tolist()]
            if fields:
                table_defs[table] = fields
                definition_sources[table] = str(path.relative_to(IMAGEWG_DIR))
                break

for table in ["image_occurrence", "image_feature"]:
    print(table, "verified:", table in table_defs)
    if table in table_defs:
        print(" source:", definition_sources[table])
        print(" fields:", table_defs[table])


image_occurrence verified: False
image_feature verified: False


In [8]:

# Gate the exact schema discovery.
schema_verified = all(t in table_defs and len(table_defs[t]) > 0
                      for t in ["image_occurrence", "image_feature"])

print("Exact ImageWG table schema verified from pinned revision:", schema_verified)

if not schema_verified:
    print(
        "\nSTOP CONDITION: The current pinned ImageWG repository revision identifies the MI-CDM tables "
        "but this notebook could not recover an exact field-level schema from machine-readable/explicit DDL. "
        "No Arm A rows will be fabricated."
    )


Exact ImageWG table schema verified from pinned revision: False

STOP CONDITION: The current pinned ImageWG repository revision identifies the MI-CDM tables but this notebook could not recover an exact field-level schema from machine-readable/explicit DDL. No Arm A rows will be fabricated.



## Faithful synthetic Arm A mapping

This section executes **only if exact table fields were recovered from the pinned ImageWG revision**.

The mapper is deliberately conservative:
- it maps only source dimensions that have an explicitly verified compatible target field;
- it does not manufacture Athena concept IDs;
- source codes/identifiers remain synthetic;
- unsupported dimensions are recorded as `NOT_MAPPED_BY_VERIFIED_SCHEMA`, not silently discarded;
- the mapping is a synthetic dataframe construction, **not a production database load**.


In [9]:

with open(PHASE26_BUNDLE, "r", encoding="utf-8") as f:
    fhir_bundle = json.load(f)

def bundle_index(bundle):
    idx = {}
    for e in bundle.get("entry", []):
        r = e.get("resource", {})
        if r.get("resourceType") and r.get("id"):
            idx[f"{r['resourceType']}/{r['id']}"] = r
    return idx

idx = bundle_index(fhir_bundle)
observations = [r for r in idx.values() if r.get("resourceType") == "Observation"]
studies = [r for r in idx.values() if r.get("resourceType") == "ImagingStudy"]
patients = [r for r in idx.values() if r.get("resourceType") == "Patient"]

print("FHIR source counts:", {
    "Patient": len(patients),
    "ImagingStudy": len(studies),
    "Observation": len(observations)
})
assert len(observations) == 16


FHIR source counts: {'Patient': 2, 'ImagingStudy': 4, 'Observation': 16}


In [10]:

# Generic exact-field matcher: maps known evidence meanings only to verified target fields.
def choose_field(columns, candidates):
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

mapping_contract = {
    "image_occurrence": {
        "primary_id": choose_field(table_defs.get("image_occurrence", []),
                                   ["image_occurrence_id"]),
        "person_id": choose_field(table_defs.get("image_occurrence", []),
                                  ["person_id"]),
        "date": choose_field(table_defs.get("image_occurrence", []),
                             ["image_occurrence_date","image_date"]),
        "datetime": choose_field(table_defs.get("image_occurrence", []),
                                 ["image_occurrence_datetime","image_datetime"]),
        "source_value": choose_field(table_defs.get("image_occurrence", []),
                                     ["image_occurrence_source_value","image_source_value"]),
    },
    "image_feature": {
        "primary_id": choose_field(table_defs.get("image_feature", []),
                                   ["image_feature_id"]),
        "image_occurrence_id": choose_field(table_defs.get("image_feature", []),
                                            ["image_occurrence_id"]),
        "person_id": choose_field(table_defs.get("image_feature", []),
                                  ["person_id"]),
        "value_number": choose_field(table_defs.get("image_feature", []),
                                     ["value_as_number","feature_value","image_feature_value"]),
        "source_value": choose_field(table_defs.get("image_feature", []),
                                     ["image_feature_source_value","feature_source_value"]),
    }
}

print(json.dumps(mapping_contract, indent=2))


{
  "image_occurrence": {
    "primary_id": null,
    "person_id": null,
    "date": null,
    "datetime": null,
    "source_value": null
  },
  "image_feature": {
    "primary_id": null,
    "image_occurrence_id": null,
    "person_id": null,
    "value_number": null,
    "source_value": null
  }
}


In [11]:

# Build synthetic Arm A only if exact schema is verified and minimally usable.
armA_executed = False
image_occ_df = pd.DataFrame()
image_feature_df = pd.DataFrame()
unmapped_rows = []

minimum_ok = (
    schema_verified and
    mapping_contract["image_occurrence"]["primary_id"] is not None and
    mapping_contract["image_feature"]["primary_id"] is not None and
    mapping_contract["image_feature"]["image_occurrence_id"] is not None
)

if minimum_ok:
    # Stable synthetic person IDs.
    person_map = {
        p["id"]: i+1
        for i,p in enumerate(sorted(patients, key=lambda x: x["id"]))
    }
    study_map = {
        s["id"]: i+1
        for i,s in enumerate(sorted(studies, key=lambda x: x["id"]))
    }

    occ_rows = []
    for s in studies:
        row = {c: None for c in table_defs["image_occurrence"]}
        f = mapping_contract["image_occurrence"]
        row[f["primary_id"]] = study_map[s["id"]]

        pid = s.get("subject",{}).get("reference","").split("/")[-1]
        if f["person_id"]:
            row[f["person_id"]] = person_map.get(pid)
        if f["datetime"]:
            row[f["datetime"]] = s.get("started")
        if f["date"]:
            row[f["date"]] = (s.get("started") or "")[:10] or None
        if f["source_value"]:
            row[f["source_value"]] = s["id"]

        occ_rows.append(row)

    feat_rows = []
    for i,o in enumerate(sorted(observations, key=lambda x: x["id"]), start=1):
        row = {c: None for c in table_defs["image_feature"]}
        f = mapping_contract["image_feature"]
        row[f["primary_id"]] = i

        study_ref = o.get("focus",[{}])[0].get("reference","")
        sid = study_ref.split("/")[-1]
        row[f["image_occurrence_id"]] = study_map.get(sid)

        pid = o.get("subject",{}).get("reference","").split("/")[-1]
        if f["person_id"]:
            row[f["person_id"]] = person_map.get(pid)
        if f["value_number"]:
            row[f["value_number"]] = o.get("valueQuantity",{}).get("value")
        if f["source_value"]:
            row[f["source_value"]] = o.get("code",{}).get("text")

        # Explicitly account for dimensions not proven to have exact target fields.
        evidence_dims = {
            "unit": o.get("valueQuantity",{}).get("unit"),
            "anatomy_laterality": (
                o.get("bodySite",{}).get("coding",[{}])[0].get("code")
            ),
            "device_reference": o.get("device",{}).get("reference"),
            "trace_state": next(
                (e.get("valueCode") for e in o.get("extension",[])
                 if e.get("url","").endswith("trace-state")), None
            ),
            "trace_rule_version": next(
                (e.get("valueString") for e in o.get("extension",[])
                 if e.get("url","").endswith("trace-rule-version")), None
            )
        }
        for dim,val in evidence_dims.items():
            unmapped_rows.append({
                "observation_id": o["id"],
                "dimension": dim,
                "source_value_present": val is not None,
                "status": "NOT_MAPPED_BY_VERIFIED_SCHEMA"
            })

        feat_rows.append(row)

    image_occ_df = pd.DataFrame(occ_rows)
    image_feature_df = pd.DataFrame(feat_rows)

    assert len(image_occ_df) == len(studies)
    assert len(image_feature_df) == len(observations)
    assert image_feature_df[
        mapping_contract["image_feature"]["image_occurrence_id"]
    ].notna().all()

    armA_executed = True
    print("Synthetic Arm A dataframe mapping executed.")
    display(image_occ_df.head())
    display(image_feature_df.head())
else:
    print(
        "Synthetic Arm A NOT executed because the pinned schema was not sufficiently verified "
        "for a faithful mapping."
    )


Synthetic Arm A NOT executed because the pinned schema was not sufficiently verified for a faithful mapping.


In [12]:

# Save all Phase 27 artifacts to the same Drive folder.
OUT_JSON = WORKDIR / "ICHI2027_Phase27_SHAREABLE.json"
OUT_SCHEMA = WORKDIR / "ICHI2027_Phase27_PINNED_TARGET_SCHEMA_AUDIT.csv"
OUT_CONTRACT = WORKDIR / "ICHI2027_Phase27_MAPPING_CONTRACT.json"
OUT_UNMAPPED = WORKDIR / "ICHI2027_Phase27_UNMAPPED_DIMENSIONS.csv"
OUT_OCC = WORKDIR / "ICHI2027_Phase27_SYNTHETIC_IMAGE_OCCURRENCE.csv"
OUT_FEAT = WORKDIR / "ICHI2027_Phase27_SYNTHETIC_IMAGE_FEATURE.csv"

schema_inventory.to_csv(OUT_SCHEMA, index=False)

with open(OUT_CONTRACT, "w", encoding="utf-8") as f:
    json.dump({
        "omop_target": "OMOP CDM v5.5",
        "imagewg_commit": commit,
        "definition_sources": definition_sources,
        "verified_table_columns": table_defs,
        "semantic_mapping_contract": mapping_contract,
    }, f, indent=2)

pd.DataFrame(unmapped_rows).to_csv(OUT_UNMAPPED, index=False)

if armA_executed:
    image_occ_df.to_csv(OUT_OCC, index=False)
    image_feature_df.to_csv(OUT_FEAT, index=False)

shareable = {
    "project": "ICHI2027_neuroimaging_fhir_omop_fidelity",
    "phase": "27_pinned_target_contract_and_synthetic_armA_gate",
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "privacy": "Synthetic-only execution; no ADNI records, RIDs, private participant identifiers, or real imaging identifiers.",
    "phase26_sha256": sha256_file(PHASE26_JSON),
    "phase26_bundle_sha256": sha256_file(PHASE26_BUNDLE),
    "targets": {
        "omop_cdm": "v5.5",
        "imagewg_repository": "https://github.com/OHDSI/ImageWG",
        "imagewg_pinned_commit": commit,
        "micdm_tables_expected": ["image_occurrence","image_feature"],
        "exact_table_schema_verified_from_pinned_revision": schema_verified,
        "definition_sources": definition_sources
    },
    "arm_registry": {
        "A": {
            "name": "Faithful standards-supported mapping",
            "status": "SYNTHETIC_DATAFRAME_MAPPING_EXECUTED" if armA_executed else "BLOCKED_BY_UNVERIFIED_TARGET_SCHEMA",
            "database_load_completed": False
        },
        "B": {"status": "NOT_EXECUTED"},
        "C": {
            "status": "NOT_EXECUTABLE_ON_MRI_AS_IS",
            "executed": False
        },
        "D": {"status": "NOT_EXECUTED_IN_PHASE27"}
    },
    "mapping": {
        "synthetic_fhir_observation_count": len(observations),
        "synthetic_imaging_study_count": len(studies),
        "image_occurrence_rows_created": int(len(image_occ_df)) if armA_executed else 0,
        "image_feature_rows_created": int(len(image_feature_df)) if armA_executed else 0,
        "unmapped_dimension_events_recorded": int(len(unmapped_rows))
    },
    "gates": {
        "phase26_verified": True,
        "omop_v55_target_frozen": True,
        "imagewg_commit_frozen": True,
        "exact_MICDM_schema_verified": bool(schema_verified),
        "synthetic_armA_mapping_executed": bool(armA_executed),
        "synthetic_armA_database_load_completed": False,
        "pyOMOP_ETL_executed": False,
        "official_MICDM_production_load_completed": False,
        "Athena_mappings_approved": False,
        "real_MRI_rows_processed": 0,
        "true_four_arm_comparison_completed": False,
        "OMOP_target_cohort_equivalence_tested": False
    },
    "next_stage_gate": (
        "If exact MI-CDM schema and synthetic Arm A are verified, validate cardinality, linkage, "
        "idempotence, and source-versus-target synthetic cohort equivalence before constructing Arm B or D. "
        "If schema verification is blocked, resolve the official ImageWG field specification first; "
        "do not invent fields."
    )
}

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(shareable, f, indent=2)

print("Saved:")
for p in [OUT_JSON, OUT_SCHEMA, OUT_CONTRACT, OUT_UNMAPPED]:
    print(" -", p)
if armA_executed:
    print(" -", OUT_OCC)
    print(" -", OUT_FEAT)


Saved:
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase27_SHAREABLE.json
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase27_PINNED_TARGET_SCHEMA_AUDIT.csv
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase27_MAPPING_CONTRACT.json
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase27_UNMAPPED_DIMENSIONS.csv


In [13]:

# Final readback gate
with open(OUT_JSON, "r", encoding="utf-8") as f:
    final = json.load(f)

assert final["gates"]["phase26_verified"] is True
assert final["gates"]["omop_v55_target_frozen"] is True
assert final["gates"]["imagewg_commit_frozen"] is True
assert final["gates"]["pyOMOP_ETL_executed"] is False
assert final["gates"]["official_MICDM_production_load_completed"] is False
assert final["gates"]["real_MRI_rows_processed"] == 0
assert final["arm_registry"]["C"]["executed"] is False

print("PHASE 27 COMPLETE")
print("Exact MI-CDM schema verified :", final["gates"]["exact_MICDM_schema_verified"])
print("Synthetic Arm A executed     :", final["gates"]["synthetic_armA_mapping_executed"])

if final["gates"]["synthetic_armA_mapping_executed"]:
    print("NEXT: Phase 28 — synthetic target linkage/cardinality/idempotence + cohort-equivalence testing.")
else:
    print("NEXT: resolve the exact official ImageWG field specification before Arm A proceeds.")


PHASE 27 COMPLETE
Exact MI-CDM schema verified : False
Synthetic Arm A executed     : False
NEXT: resolve the exact official ImageWG field specification before Arm A proceeds.
